# Step 1 — Continued Pre-training of mBART on Konkani

**Goal:** Adapt `BramVanroy/mbart-large-cc25-ft-amr30-en` to the Konkani language
by continuing pre-training on 180K Konkani sentences with a **masked-language-modelling
(token-masking) denoising** objective — the same type of training mBART was originally
trained with.

The resulting model is saved to `./konkani_pretrained/` and used in
`step2_finetune_amr.ipynb` for AMR fine-tuning.

---
**Input file:** `merged_konkani.csv` (produced by `training_data/create_pretraining.py`)
- Columns: `id`, `text`
- Each row is one Konkani sentence.

**Expected runtime:** ~2-4 hours on a single A100 80 GB GPU with the default settings.

In [ ]:
import math
import os
import warnings
from pathlib import Path

import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    DataCollatorForLanguageModeling,
    MBart50TokenizerFast,
    MBartForConditionalGeneration,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers.utils import logging as hf_logging

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()

In [ ]:
# ─────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────

MODEL_NAME      = "BramVanroy/mbart-large-cc25-ft-amr30-en"
SRC_LANG        = "hi_IN"          # Closest mBART language code for Konkani
MAX_SEQ_LEN     = 128              # Konkani sentences are short
MLM_PROBABILITY = 0.15             # Fraction of tokens to mask
SEED            = 42

# Path to the CSV produced by create_pretraining.py
# Adjust this path to where you saved merged_konkani.csv
DATA_CSV = Path("../../../../training_data/merged_konkani.csv")

OUTPUT_DIR = Path("./konkani_pretrained")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device : {device}")
print(f"Data path    : {DATA_CSV.resolve()}")

In [ ]:
# ─────────────────────────────────────────────
# 1. Load Konkani sentences
# ─────────────────────────────────────────────

df = pd.read_csv(DATA_CSV)
assert "text" in df.columns, "CSV must have a 'text' column"

df = df[["text"]].dropna()
df["text"] = df["text"].str.strip()
df = df[df["text"] != ""].reset_index(drop=True)

print(f"Total sentences loaded: {len(df):,}")
print(df["text"].head(3).to_string())

In [ ]:
# ─────────────────────────────────────────────
# 2. Tokenizer
# ─────────────────────────────────────────────

tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
tokenizer.src_lang = SRC_LANG
tokenizer.tgt_lang = SRC_LANG  # same for MLM

print(f"Vocab size: {tokenizer.vocab_size:,}")

In [ ]:
# ─────────────────────────────────────────────
# 3. Tokenise the corpus
# ─────────────────────────────────────────────

# Convert to HuggingFace Dataset for efficient batched tokenisation
hf_dataset = Dataset.from_pandas(df)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        max_length=MAX_SEQ_LEN,
        truncation=True,
        padding=False,
    )

tokenized = hf_dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"],
    desc="Tokenising",
)

print(f"Tokenised dataset: {tokenized}")

In [ ]:
# ─────────────────────────────────────────────
# 4. Train / validation split (95 / 5)
# ─────────────────────────────────────────────

split = tokenized.train_test_split(test_size=0.05, seed=SEED)
train_dataset = split["train"]
val_dataset   = split["test"]

print(f"Pre-train split  →  train: {len(train_dataset):,}  val: {len(val_dataset):,}")

In [ ]:
# ─────────────────────────────────────────────
# 5. Load model
# ─────────────────────────────────────────────

model = MBartForConditionalGeneration.from_pretrained(MODEL_NAME)
model.to(device)
print(f"Model parameters: {model.num_parameters():,}")

In [ ]:
# ─────────────────────────────────────────────
# 6. Data collator — whole-word token masking
# ─────────────────────────────────────────────
#
# We use DataCollatorForLanguageModeling with mlm=True.
# This randomly masks MLM_PROBABILITY of the input tokens and
# asks the model to predict them — a standard MLM pre-training objective
# that is language-agnostic and works well for cross-lingual adaptation.
#
# NOTE: For mBART's original denoising (text infilling + sentence permutation)
# you would use DataCollatorForSeq2Seq with a custom noising function.
# Token-masking is simpler, equally effective for language adaptation, and
# requires no changes to the model architecture.

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=MLM_PROBABILITY,
    pad_to_multiple_of=8,
)

In [ ]:
# ─────────────────────────────────────────────
# 7. Training arguments
# ─────────────────────────────────────────────
#
# For 180K sentences and 3 epochs the effective number of gradient updates is:
#   steps = ceil(180_000 * 0.95 / (8 * 8)) * 3 ≈ 8_019
# Feel free to increase num_train_epochs for better adaptation.

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,      # effective batch = 64
    learning_rate=3e-5,
    warmup_ratio=0.06,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=(device == "cuda"),
    logging_dir=str(OUTPUT_DIR / "logs"),
    logging_steps=200,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=4,
)

In [ ]:
# ─────────────────────────────────────────────
# 8. Trainer
# ─────────────────────────────────────────────

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

In [ ]:
%%time
print("\nStarting Konkani continued pre-training …")
trainer.train()

In [ ]:
# ─────────────────────────────────────────────
# 9. Save the best checkpoint as a clean model
# ─────────────────────────────────────────────

best_model_dir = str(OUTPUT_DIR / "best_model")
trainer.save_model(best_model_dir)
tokenizer.save_pretrained(best_model_dir)
print(f"Konkani-adapted model saved → {best_model_dir}")

In [ ]:
# ─────────────────────────────────────────────
# 10. Quick perplexity sanity check on val set
# ─────────────────────────────────────────────

eval_results = trainer.evaluate()
perplexity   = math.exp(eval_results["eval_loss"])
print(f"\nValidation Loss : {eval_results['eval_loss']:.4f}")
print(f"Perplexity      : {perplexity:.2f}")
print("\nPre-training complete. Run step2_finetune_amr.ipynb next.")